# MobileCLIP2-S0 on CINIC-10

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/A3_Recv/blob/main/finished/colabs/05_mobileclip_s0_cinic.ipynb)

PE gave the best accuracy, but its visual encoder has about 1.88B parameters. I wanted to see how much accuracy I would lose with a model of only about **11.4M parameters**, so I fine-tuned the MobileCLIP2-S0 visual encoder and a 10-class head on CINIC-10.

`submission_mobileclip2_s0_cinic.csv` reached **0.84364** on Kaggle. This is **12.412 percentage points** below PE, with roughly **165× fewer visual/classifier parameters**. The training data and adaptation method also change, so this is a practical size/accuracy comparison rather than an isolated scaling experiment.

The recorded run used an NVIDIA L4. Checkpoints and logs are saved in Drive.


## 1. Data and checkpoint selection

| Split | Use |
|---|---|
| CINIC-10 train | Supervised training: 90,000 images before deduplication |
| CINIC-10 validation | Selection subset of up to 10,000 images, 1,000 per class |
| Remaining CINIC validation and CINIC test | Unused |
| Competition validation | Evaluation after selecting the checkpoint |
| Competition test | Final inference |

CINIC combines CIFAR and ImageNet images. I use CINIC training data alone, without adding the competition training split again.

Before training, exact RGB duplicates and horizontal mirrors of CINIC validation and competition validation images are removed from the training set. The selection subset is also checked against competition validation. This leaves **87,804 training images** in the recorded run; it is not a near-duplicate search.

I select the checkpoint by CINIC validation accuracy, breaking ties by cross-entropy. Competition validation is evaluated afterward and has already been used in earlier experiments.


## 2. Dependencies and setup

OpenCLIP and timm provide the S0 architecture and checkpoint adapter. Keep Colab's paired PyTorch/torchvision installation. The checkpoint is public and does not need a Hugging Face token.


In [ ]:
%pip -q install "open_clip_torch==3.2.0" "timm==1.0.22" "huggingface_hub==0.36.0" "kagglehub==1.0.3" "pyarrow==21.0.0" matplotlib


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, random, sys, time
import requests, numpy as np, torch
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
RUN_NAME = 'mobileclip2_s0_cinic_supervised_v1'
RUN_DIR = Path('/content/drive/MyDrive/A3_Recv/runs') / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path('/content/cinic10_cache')  # local SSD: faster than thousands of PNGs on Drive
SESSION_HOURS = 6.5  # training budget only; downloads/setup take additional time

ENGINE_COMMIT = 'd78a1ad6430cd78b28298aba3e9ece50c846a436'
ENGINE_SHA256 = '5b524949368f0f21890d6fcfab864c24ac7da8e4b142cf7078c4b53a700ad7d7'
url = f'https://raw.githubusercontent.com/rivas-vega/A3_Recv/{ENGINE_COMMIT}/mobileclip_s0_cinic.py'
r = requests.get(url, timeout=60); r.raise_for_status()
assert hashlib.sha256(r.content).hexdigest() == ENGINE_SHA256, 'Experiment module hash mismatch'
engine_path = Path('/content/mobileclip_s0_cinic.py'); engine_path.write_bytes(r.content)
spec = importlib.util.spec_from_file_location('s0_cinic', engine_path)
m = importlib.util.module_from_spec(spec); sys.modules[spec.name] = m; spec.loader.exec_module(m)
CFG = dict(m.DEFAULTS)
m.seed_all(CFG['seed'])
assert torch.cuda.is_available(), 'Colab → Runtime → Change runtime type → GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print('Results:', RUN_DIR)
print(json.dumps(CFG, indent=2))


## 3. Kaggle credentials

I store `KAGGLE_API_TOKEN` in Colab Secrets and enable notebook access. The legacy `KAGGLE_USERNAME` / `KAGGLE_KEY` pair is also supported. Credentials stay outside the notebook.

`COMPETITION_DIR` can point to an existing download. At this stage, the code reads competition validation; test images are opened during the final export.


In [ ]:
import kagglehub
COMPETITION_DIR = ''  # optional path to an existing download
if COMPETITION_DIR:
    DATA_ROOT = Path(COMPETITION_DIR)
else:
    m.configure_kaggle_secrets()  # read credentials from Colab Secrets
    try:
        DATA_ROOT = Path(kagglehub.competition_download('mva-recvis-2026'))
    except Exception as exc:
        if 'auth' in type(exc).__name__.lower() or '401' in str(exc) or '403' in str(exc):
            raise RuntimeError('Kaggle access denied. Check the secret and confirm the account has '
                               'accepted the mva-recvis-2026 rules.') from None
        raise
audit_x, audit_y, audit_sha = m.read_validation(DATA_ROOT)
print('Competition validation:', audit_x.shape, 'Loaded from competition archive')


## 4. Download and prepare CINIC-10

The original Edinburgh download returned HTTP 403 in Colab, so I used the public [Flower mirror](https://huggingface.co/datasets/flwrlabs/cinic10), pinned by revision and file hashes.

The 180,000 train/validation filenames, decoded RGB images and labels were checked against the original archive. Only those two splits are downloaded. The Parquet images are decoded to 32×32 uint8 arrays and cached on the local disk.

The selection sample uses seed 42. The manifest records source hashes, class mapping and duplicate exclusions.


In [ ]:
raw, cinic_info = m.download_cinic(CACHE_DIR)
data, data_identity = m.prepare_data(raw, audit_x, audit_y, CFG, RUN_DIR,
    provenance={'cinic': cinic_info, 'competition_validation_sha256': audit_sha})
del raw, audit_x, audit_y
import gc; gc.collect()
print(json.dumps(data_identity['counts'], indent=2))
for key in ('train_y', 'selection_y'):
    print(key, dict(zip(m.CLASSES, np.bincount(data[key], minlength=10).tolist())))


## 5. Model

**32×32 RGB → official 256×256 preprocessing → FastViT MCI0 visual encoder → 512-dimensional feature → 10-class linear head.**

S0 combines convolutions and attention. I discard the text tower and initialize a new classifier head. The loaded model has **11,406,976 visual parameters** plus **5,130 head parameters**.

Training uses the CINIC class labels. MobileCLIP2's original pretraining used teachers, but I do not perform additional distillation here.

The official preprocessing uses bilinear resize and mean 0 / standard deviation 1. I keep that normalization and leave the branches unfused during training.


In [ ]:
model, train_tf, eval_tf, model_identity = m.load_model(RUN_DIR, CFG)
print('Official preprocessing:', eval_tf)
print('Training augmentation:', train_tf)
print('Visual parameters:', f"{model_identity['visual_parameters']:,}")
print('Head parameters:', f"{model_identity['classifier_parameters']:,}")
assert model_identity['model_config']['vision_cfg']['timm_model_name'] == 'fastvit_mci0'
assert model_identity['model_config']['embed_dim'] == 512


## 6. Inspect the training transform

Four random training images: original 32×32, official 256×256 preprocessing and an augmented training view. The augmentation is a mild crop with two-pixel reflect padding, horizontal flip and moderate color jitter, applied before resizing.


In [ ]:
chosen = np.random.default_rng(CFG['seed']).choice(len(data['train_x']), 4, replace=False)
fig, axes = plt.subplots(4, 3, figsize=(10, 11))
mean = torch.tensor(model_identity['preprocess']['mean'])[:, None, None]
std = torch.tensor(model_identity['preprocess']['std'])[:, None, None]
for row, i in enumerate(chosen):
    im = m.Image.fromarray(data['train_x'][i])
    views = [np.asarray(im), (eval_tf(im)*std+mean).permute(1,2,0).clamp(0,1).numpy(),
             (train_tf(im)*std+mean).permute(1,2,0).clamp(0,1).numpy()]
    for col, (view, title) in enumerate(zip(views, ['Original 32×32', 'Bilinear 256×256', 'Training view'])):
        axes[row,col].imshow(view, interpolation='nearest'); axes[row,col].axis('off')
        axes[row,col].set_title(f'{title} — {m.CLASSES[int(data["train_y"][i])]}')
plt.tight_layout(); plt.show()


## 7. Training configuration

- **Epochs 1–3:** head only, AdamW, LR `1e-3`; encoder frozen in evaluation mode.
- **Epochs 4–25:** full encoder LR `1e-5`, head LR `1e-4`, cosine decay. BatchNorm running statistics stay frozen; affine parameters can learn.
- Cross-entropy with label smoothing 0.05, weight decay 0.01 and gradient clipping at 1.
- Early stopping after five fine-tuning epochs without improvement, once at least five fine-tuning epochs have completed.
- Best checkpoint by CINIC validation accuracy, then loss. BF16 when supported, otherwise FP16 with GradScaler.

The protocol and versions are saved before training. Resumption is at epoch boundaries, so an interrupted epoch is repeated.


In [ ]:
import importlib.metadata as metadata
VERSIONS = {name: metadata.version(name) for name in
    ['torch', 'torchvision', 'open_clip_torch', 'timm', 'huggingface_hub', 'numpy', 'Pillow']}
PROTOCOL = dict(config=CFG, classes=list(m.CLASSES), engine_sha256=ENGINE_SHA256,
    model=model_identity, data=data_identity, versions=VERSIONS,
    selection='CINIC valid accuracy, tie: cross entropy',
    audit='competition validation, after selection only',
    inference='single image, official preprocessing, no TTA',
    supervision='true class labels only; no additional distillation')
m.lock_json(PROTOCOL, RUN_DIR/'protocol.json')
model = model.to('cuda')
sample = torch.stack([eval_tf(m.Image.fromarray(x)) for x in data['train_x'][:2]]).cuda()
model.eval()
with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16):
    logits = model(sample)
assert logits.shape == (2, 10) and torch.isfinite(logits).all()
del sample, logits
print('Model load and forward pass OK; protocol:', m.json_hash(PROTOCOL))


## 8. Train or resume

The training call has a 6.5-hour budget, checked between epochs. To resume, keep the same `RUN_NAME` and configuration. `last.pt` stores progress and optimizer state; `best.pt` stores the selected checkpoint.

My recorded L4 run completed 25 epochs, taking about **3.61 hours of training**. Runtime depends on the GPU and data loading.


In [ ]:
history = m.fit(model, data, train_tf, eval_tf, CFG, RUN_DIR, PROTOCOL, session_hours=SESSION_HOURS)
if (RUN_DIR/'status.json').exists():
    print((RUN_DIR/'status.json').read_text())


In [ ]:
history = json.loads((RUN_DIR/'history.json').read_text())
fig, axes = plt.subplots(1,2,figsize=(11,4))
epochs = [r['epoch'] for r in history]
axes[0].plot(epochs,[100*r['selection_accuracy'] for r in history],marker='.')
axes[0].set_ylabel('Accuracy CINIC valid (%)')
axes[1].plot(epochs,[r['train_loss'] for r in history],label='Train (label smoothing)')
axes[1].plot(epochs,[r['selection_loss'] for r in history],label='Selection (no smoothing)')
axes[1].legend(); axes[1].set_ylabel('Cross-entropy')
for ax in axes:
    ax.axvline(CFG['warmup_epochs']+.5,color='gray',ls='--'); ax.set_xlabel('Epoch'); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()


## 9. Evaluate the selected checkpoint

Once training is complete, load the checkpoint selected on CINIC validation and evaluate it on competition validation. This evaluation does not change checkpoint selection.

The recorded accuracies were **94.78% on CINIC validation** and **97.99% on competition validation**, compared with **84.364% on Kaggle**. The difference again shows why clean validation was not a reliable estimate of synthetic test performance.


In [ ]:
best = m.load_selected(model, RUN_DIR, PROTOCOL)
device = next(model.parameters()).device
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
audit_dl = m.loader(data['audit_x'], data['audit_y'], eval_tf, CFG)
audit_metrics, audit_probs = m.evaluate(model, audit_dl, device, amp_dtype)
report = dict(selected_checkpoint=best, competition_validation=audit_metrics,
    status=json.loads((RUN_DIR/'status.json').read_text()),
    total_parameters=sum(p.numel() for p in model.parameters()),
    checkpoint_megabytes=(RUN_DIR/'best.pt').stat().st_size/2**20,
    gpu=torch.cuda.get_device_name(0), max_allocated_vram_gb=torch.cuda.max_memory_allocated()/2**30,
    note='No Kaggle score computed; audit did not select the model')
m.atomic_json(report, RUN_DIR/'audit_report.json')
print(json.dumps(report, indent=2))
truth=data['audit_y']; pred=audit_probs.argmax(1)
conf=np.zeros((10,10),dtype=np.int64); np.add.at(conf,(truth,pred),1)
normalized=conf/np.maximum(conf.sum(1,keepdims=True),1)
fig,ax=plt.subplots(figsize=(8,7)); im=ax.imshow(normalized,vmin=0,vmax=1,cmap='Blues')
ax.set_xticks(range(10),m.CLASSES,rotation=45,ha='right');ax.set_yticks(range(10),m.CLASSES)
ax.set_xlabel('Prediction');ax.set_ylabel('True label');ax.set_title('Competition validation after selection')
fig.colorbar(im,ax=ax);plt.tight_layout();plt.show()


## 10. Export the submission

`EXPORT_FINAL=True` exports predictions after training and checkpoint selection. Set it to `False` when only inspecting training results.

The exporter records the checkpoint hash, reads test images and IDs, aligns predictions with `sample_submission.csv` and saves an integrity record. Inference uses one full-image view with the official preprocessing.


In [ ]:
EXPORT_FINAL = True
if EXPORT_FINAL:
    csv_path = m.export_final(model, DATA_ROOT, eval_tf, CFG, RUN_DIR, PROTOCOL)
    print('Submission CSV:', csv_path)
    from google.colab import files
    files.download(str(csv_path))
else:
    print('Export disabled; test images have not been loaded.')


## Notes and references

This run trades accuracy for a much smaller encoder. It uses a different training dataset and supervised fine-tuning, so I cannot attribute the entire gap from PE to parameter count. I would need a controlled comparison to separate the effects of model size, pretraining and adaptation.

- [Apple MobileCLIP2-S0](https://huggingface.co/apple/MobileCLIP2-S0).
- [OpenCLIP checkpoint revision](https://huggingface.co/timm/MobileCLIP2-S0-OpenCLIP/tree/095906d28bf54d7584dc411e8ffe448f34289e05).
- [CINIC-10 documentation](https://github.com/BayesWatch/cinic-10).
- [Pinned Flower mirror](https://huggingface.co/datasets/flwrlabs/cinic10/tree/44c23bab233087df95aad862c5b6fa0e5e6030cf).
- [Original CINIC-10 archive](https://datashare.is.ed.ac.uk/handle/10283/3192).
